# Compare Benchmark Runs
This notebook demonstrates how you can analyze the differences between two benchmark runs of the same benchmark and find the tests that differ the most, which probably means that they require further analysis to figure out why they changed.

Several projects exist in the `examples` folder, but this notebook assumes we are working on the
JVM part of the `kotlin-multiplatform` project. But the same approach can be used for the other projects.

First, you need to run the benchmark twice. This can be done by running these commands from the root of the project:

```shell
> ./gradlew :examples:kotlin-multiplatform:jvmBenchmark
> ./gradlew :examples:kotlin-multiplatform:jvmBenchmark
```

Once it is completed, run this notebook, and it will automatically find the latest result.

In [1]:
%use serialization, dataframe, kandy

In [2]:
// Serialization classes matching the JMH-alike JSON format.
// We define these classes manually so we can keep `params` as a JsonObject, as it means we can handle them
// in a generic manner. If you benchmark have fixed params, using `"<jsonText>".deserializeThis()` is
// faster and easier.

@Serializable
public data class Benchmark(
    public val benchmark: String,
    public val mode: String,
    public val warmupIterations: Int,
    public val warmupTime: String,
    public val measurementIterations: Int,
    public val measurementTime: String,
    public val primaryMetric: PrimaryMetric,
    public val secondaryMetrics: Map<String, PrimaryMetric>,
    public val params: JsonObject? = null
)

@Serializable
public data class PrimaryMetric(
    public val score: Double,
    public val scoreError: Double,
    public val scoreConfidence: List<Double>,
    public val scorePercentiles: Map<String, Double>,
    public val scoreUnit: String,
    public val rawData: List<List<Double>>,
)

In [3]:
import java.nio.file.Files
import java.nio.file.attribute.BasicFileAttributes
import kotlin.io.path.exists
import kotlin.io.path.forEachDirectoryEntry
import kotlin.io.path.isDirectory
import kotlin.io.path.listDirectoryEntries
import kotlin.io.path.readText

// Find latest result file, based on the their timestamp.
val runsDir = notebook.workingDir.resolve("build/reports/benchmarks/main")
val outputFiles = runsDir.listDirectoryEntries()
    .filter { it.isDirectory() }
    .sortedByDescending { dir -> Files.readAttributes(dir, BasicFileAttributes::class.java).creationTime() }
    .subList(0, 2)
    .map { it.resolve("jvm.json") }

In [4]:
// Convert to typed JSON
val json = Json { ignoreUnknownKeys = true }
val newRun = json.decodeFromString<List<Benchmark>>(outputFiles.first().readText())
val oldRun = json.decodeFromString<List<Benchmark>>(outputFiles.last().readText())

In [5]:
// Convert to DataFrames for easier processing. As there is not "id" keys for the benchmark, we invent one by just
// assigning the test row index as their "primary key". We could attempt to use the benchmark name and param values,
// but that is complicated by how paramers are represented in the JSON file. So, since we assume that the two files
// are equal using row index should be safe.
val oldDf = oldRun.toDataFrame().addId("rowIndex")
val newDf = newRun.toDataFrame().addId("rowIndex")

In [6]:
val combinedData = oldDf.innerJoin(newDf) { rowIndex }
// Un-commont this to see the intermediate dataframe:
// combinedData

In [7]:
import kotlinx.serialization.json.encodeToJsonElement

// Reduce the combined data into the exact format we need
val resultData = combinedData.mapToFrame {
    "name" from { it.benchmark }
    "params" from {
        it.params?.entries.orEmpty()
            .sortedBy { it.key }
            .joinToString(",") { entry -> "${entry.key}=${entry.value.jsonPrimitive.content}" }
    }
    "mode" from { it.mode } // "avgt" or "thrpt"
    "unit" from { it.primaryMetric.scoreUnit }
    "runOld" {
        "score" from { it.primaryMetric.score }
        "range" from { it.primaryMetric.scoreConfidence[0]..it.primaryMetric.scoreConfidence[1] }
    }
    "runNew" {
        "score" from { it.primaryMetric1.score }
        "range" from { it.primaryMetric1.scoreConfidence[0]..it.primaryMetric1.scoreConfidence[1] }
    }
}
// Un-commont this to see the intermediate dataframe:
// resultData

In [8]:
// Flatten the data so it is easier to plot
val mergedData = resultData.unfold { runOld and runNew }.flatten()
// Un-commont this to see the intermediate dataframe:
// mergedData

In [9]:
// Before plotting the data, we calculate the change between the two runs. This is saved
// in "scoreDiff". This is done slightly different depending on the test mode:
//
// - "avgt": For the average time we use "oldScore - newScore", so improvements in the
//   benchmark result in positive numbers.
// - "thrpt": For throughput, we use "newScore - oldScore", so improvements here also
//   result in positive numbers.
//
// We also normalize this value as a percentage change from `scoreOld`. This is saved in
// "scoreDiffPercentage".
val plotData = mergedData
    .add("diffScore") {
        when (mode) {
            "avgt" -> score - score1
            "thrpt" -> score1 - score
            else -> error("Unknown mode: $mode")
        }
    }
    .add("diffScorePercentage") {
        (get("diffScore") as Double) * 100.0 / score
    }
    .add("testLabel") {
        if (params.isNullOrBlank()) {
            name
        } else {
            "$name\n[$params]"
        }
    }
    .add("barColor") {
        val value = get("diffScorePercentage") as Double
        if (value < 0.0) "neg" else "pos"
    }
plotData

name,params,mode,unit,score,range,score1,range1,diffScore,diffScorePercentage,testLabel,barColor
com.the3moly.benchmark.Transformation...,length=111,thrpt,ops/s,1343629.193865,1024204.3129925407..1663054.0747367037,1332675.154096,1278685.7149622394..1386664.5932299173,-10954.039769,-0.815258,com.the3moly.benchmark.Transformation...,neg
com.the3moly.benchmark.Transformation...,length=1111,thrpt,ops/s,88962.557664,70374.61808978496..107550.49723848351,88908.866043,79514.45972643858..98303.27235973989,-53.691621,-0.060353,com.the3moly.benchmark.Transformation...,neg
com.the3moly.benchmark.Transformation...,length=111,thrpt,ops/s,443679.787130,259985.09182090833..627374.4824388524,476032.585489,472123.8305822698..479941.3403963024,32352.798359,7.291925,com.the3moly.benchmark.Transformation...,pos
com.the3moly.benchmark.Transformation...,length=1111,thrpt,ops/s,28604.341294,19153.056891905915..38055.62569535208,29284.256704,26625.28049890627..31943.232908853068,679.915410,2.376966,com.the3moly.benchmark.Transformation...,pos


In [10]:
import org.jetbrains.kotlinx.kandy.util.color.Color
import org.jetbrains.letsPlot.core.spec.plotson.fill
import org.jetbrains.letsPlot.label.ggtitle
import org.jetbrains.letsPlot.scale.guideLegend
import org.jetbrains.letsPlot.scale.guides

// Now we can plot this data. First we create a basic plot just showing the difference in percent between all scores.
plotData.sortBy { diffScorePercentage }.plot {
    barsH {
        x(diffScorePercentage) {
            axis.name = "Diff %"
        }
        y(testLabel) {
            axis.name = ""
        }
        fillColor(barColor) {
            scale = categorical("neg" to Color.RED, "pos" to Color.GREEN)
            legend.type = LegendType.None
        }
        tooltips {
            line(diffScorePercentage, format = ".2f")
        }
    }
    layout {
        size = 800 to ((40 * plotData.size().nrow) + 100)
        style {
            global {
                title {
                    margin(10.0, 0.0)
                }
            }
        }
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="pkDb6C"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 var plotSpec={
"mapping":{
},
"data":{
"testLabel":["com.the3moly.benchmark.TransformationsBenchmark.sortedArrayDescendingTake\n[length=111]","com.the3moly.benchmark.TransformationsBenchmark.sortedArrayDescendingTake\n[length=1111]","com.the3moly.benchmark.TransformationsBenchmark.sortedTakeLast\n[length=1111]","com.the3moly.benchmark.TransformationsBenchmark.sortedTakeLast\n[length=111]"],
"diffScorePercentage":[-0.815257648357369,-0.06035305464991819,2.376965801348865,7.291925234794375],
"barColor":["neg","neg","pos","pos"]
},
"ggsize":{
"width":800.0,
"height":260.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"Diff %",
"limits":[null,null]
},{
"aesthetic":"y",
"discrete":true,
"name":""
},{
"aesthetic":"fill",
"values":["#ee6666","#3ba272"],
"limits":["neg","pos"],
"guide":"none"
}],
"layers":[{
"mapping":{
"x":"diffScorePercentage",
"y":"testLabel",
"fill":"barColor"
},
"stat":"identity",
"orientation":"y",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"tooltips":{
"lines":["@|@{diffScorePercentage}"],
"formats":[{
"field":"diffScorePercentage",
"format":".2f"
}],
"disable_splitting":true
},
"data":{
}
}],
"theme":{
"title":{
"margin":[10.0,0.0,10.0,0.0],
"blank":false
},
"axis_ontop":false,
"axis_ontop_y":false,
"axis_ontop_x":false
},
"data_meta":{
"series_annotations":[{
"type":"float",
"column":"diffScorePercentage"
},{
"type":"str",
"column":"testLabel"
},{
"type":"str",
"column":"barColor"
}]
},
"spec_id":"2"
};
 var containerDiv = document.getElementById("pkDb6C");
 
 var toolbar = null;
 var plotContainer = containerDiv; 
 
 var options = {
 sizing: {
 width_mode: "fixed",
 height_mode: "fixed",
 width: 800.0,
 height: 260.0
 }
 };
 var fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, -1, -1, plotContainer, options);
 if (toolbar) {
 toolbar.bind(fig);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 
 
 5 
 
 
 
 
 
 
 
 
 
 
 com.the3moly.benchmark.TransformationsBenchmark.sortedArrayDescendingTake [length=111] 
 
 
 
 
 
 
 com.the3moly.benchmark.TransformationsBenchmark.sortedArrayDescendingTake [length=1111] 
 
 
 
 
 
 
 com.the3moly.benchmark.TransformationsBenchmark.sortedTakeLast [length=1111] 
 
 
 
 
 
 
 com.the3moly.benchmark.TransformationsBenchmark.sortedTakeLast [length=111] 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Diff %

In [11]:
// Just comparing the score values is a bit simplistic as the benchmark results are actually a range: score +/- error.
// So, instead of plotting all tests, we want to focus only on the benchmarks that looks "interesting". This is
// defined as any benchmark that differ so much that the benchmark ranges do not overlap, i.e., we no longer just
// look at only the score but consider the full error range.
//
// We still use the "score" to calculate the change in percent, but now on a filtered list
fun kotlin.ranges.ClosedFloatingPointRange<kotlin.Double>.overlaps(other: ClosedFloatingPointRange<kotlin.Double>): Boolean =
    this.start <= other.endInclusive && other.start <= this.endInclusive

val interestingBenchmarks = plotData.filter {
    !it.range.overlaps(it.range1)
}
interestingBenchmarks

name,params,mode,unit,score,range,score1,range1,diffScore,diffScorePercentage,testLabel,barColor


In [15]:
// Now lets plot the interesting benchmarks, similar to before.
interestingBenchmarks.sortBy { diffScorePercentage }.plot {
    barsH {
        x(diffScorePercentage) {
            axis.name = "Diff %"
        }
        y(testLabel) {
            axis.name = ""
        }
        fillColor(barColor) {
            scale = categorical("neg" to Color.RED, "pos" to Color.GREEN)
            legend.type = LegendType.None
        }
        tooltips {
            line(diffScorePercentage, format = ".2f")
        }
    }
    layout {
        size = 800 to ((40 * interestingBenchmarks.size().nrow) + 100)
        style {
            global {
                title {
                    margin(10.0, 0.0)
                }
            }
        }
    }
}

The problem is found in one of the loaded libraries: check library renderers
java.util.NoSuchElementException: Can't create DoubleSpan: the input is empty or contains NULLs.
org.jetbrains.kotlinx.jupyter.exceptions.ReplLibraryException: The problem is found in one of the loaded libraries: check library renderers
	at org.jetbrains.kotlinx.jupyter.exceptions.CompositeReplExceptionKt.throwAsLibraryException(CompositeReplException.kt:48)
	at org.jetbrains.kotlinx.jupyter.exceptions.ReplLibraryExceptionKt.rethrowAsLibraryException(ReplLibraryException.kt:42)
	at org.jetbrains.kotlinx.jupyter.codegen.RenderersProcessorImpl.renderResult(RenderersProcessorImpl.kt:35)
	at org.jetbrains.kotlinx.jupyter.repl.impl.ReplForJupyterImpl.renderResult$lambda$0$0(ReplForJupyterImpl.kt:536)
	at org.jetbrains.kotlinx.jupyter.config.LoggingKt.catchAll(Logging.kt:33)
	at org.jetbrains.kotlinx.jupyter.config.LoggingKt.catchAll$default(Logging.kt:27)
	at org.jetbrains.kotlinx.jupyter.repl.impl.ReplForJupyterIm

null